# 02 - Backtesting: Cruce de Medias Móviles + Optimización

**Capítulo**: 02 - SMA Cross

**Objetivo**: Backtest de SMA Cross, optimización de parámetros n1/n2, y análisis de sobreajuste.

---

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import SMA

from curso.lib.data import download_historical
from curso.lib.backtest import run_backtest, extract_metrics, metrics_to_dataframe, optimize_strategy
from curso.lib.reporting import plot_equity_curve, plot_drawdown, print_metrics_table

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
print('Setup completado ✓')

## 1. Datos

In [ ]:
TICKER = 'SPY'
df = download_historical(TICKER)
print(f'{TICKER}: {len(df)} registros')

## 2. Definir estrategia

In [ ]:
class SmaCross(Strategy):
    n1 = 10
    n2 = 20
    
    def init(self):
        close = self.data.Close
        self.sma1 = self.I(SMA, close, self.n1)
        self.sma2 = self.I(SMA, close, self.n2)
    
    def next(self):
        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy()
        elif crossover(self.sma2, self.sma1):
            self.position.close()
            self.sell()

## 3. Backtest con parámetros default

In [ ]:
stats, bt = run_backtest(df, SmaCross)
metrics = extract_metrics(stats)
print_metrics_table(metrics_to_dataframe(metrics))

## 4. Optimización de parámetros

In [ ]:
# Optimizar n1 y n2
opt_stats, opt_bt = optimize_strategy(
    df, SmaCross,
    maximize='Sharpe Ratio',
    n1=range(5, 50, 5),
    n2=range(10, 100, 5),
    constraint=lambda p: p.n1 < p.n2
)

print(f'Mejores parámetros: n1={opt_stats._strategy.n1}, n2={opt_stats._strategy.n2}')
opt_metrics = extract_metrics(opt_stats)
print_metrics_table(metrics_to_dataframe(opt_metrics))

## 5. Análisis de sobreajuste

Dividimos datos en train (70%) y test (30%) para verificar robustez.

In [ ]:
split_idx = int(len(df) * 0.7)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

print(f'Train: {len(train)} registros ({train.index.min().date()} → {train.index.max().date()})')
print(f'Test: {len(test)} registros ({test.index.min().date()} → {test.index.max().date()})')

# Optimizar solo en train
train_stats, _ = optimize_strategy(train, SmaCross, n1=range(5,50,5), n2=range(10,100,5), constraint=lambda p: p.n1 < p.n2)
best_n1 = train_stats._strategy.n1
best_n2 = train_stats._strategy.n2
print(f'\nMejores params (train): n1={best_n1}, n2={best_n2}')

# Evaluar en test con esos parámetros
class SmaCrossOpt(SmaCross):
    n1 = best_n1
    n2 = best_n2

test_stats, _ = run_backtest(test, SmaCrossOpt)
print(f'\nRendimiento en TEST: {test_stats["Return [%]"]:.2f}%')
print(f'Sharpe en TEST: {test_stats["Sharpe Ratio"]:.3f}')

## 6. Conclusiones

In [ ]:
print('''
CONCLUSIONES
============
1. Rendimiento: [Comparar default vs optimizado vs test]
2. Overfitting: [¿Los resultados de test son significativamente peores que train?]
3. Robustez: [¿La estrategia funciona en un rango de parámetros?]
4. DECISIÓN: [aprobar / iterar / descartar]
''')